# Pupil-weighted orthogonality of control modes

This notebook compares the same Zernike control basis on a circular pupil and on the analytical ELT/HARMONI pupil. It then applies `PupilOrthogonalizedBasis` to recover unit-RMS, mutually orthogonal modes on the actual ELT illumination.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux import (
    CircularAperture,
    ELTHarmoniPupil,
    Grid,
    PupilOrthogonalizedBasis,
    Spectrum,
    ZernikeBasis,
)
from fiatlux.core.spectrum import Band

torch.set_default_dtype(torch.float64)

n_pixels = 161
pupil_radius = 21.0
spacing = 2 * pupil_radius / (n_pixels - 1)
grid = Grid(
    n_pixels, n_pixels, spacing, spacing, dtype=torch.float64
)
spectrum = Spectrum(
    magnitude=0,
    band=Band(1.65e-6, 0.0, 1.0),
    samples=1,
    dtype=torch.float64,
)

circular_pupil = CircularAperture(grid, radius=pupil_radius)
elt_pupil = ELTHarmoniPupil(grid)
circular_pupil.build(spectrum)
elt_pupil.build(spectrum)

print(f"Grid: {grid.shape}, sampling = {spacing:.4f} m")
print(f"ELT segments: {elt_pupil.number_of_segments}")

## The two pupil metrics

The ELT pupil includes the central obscuration and spiders but, by default, does not impose an additional outer circular clipping mask. Orthogonality is evaluated only where intensity is transmitted.

In [ ]:
extent = [
    float(grid.x.min()),
    float(grid.x.max()),
    float(grid.y.min()),
    float(grid.y.max()),
]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for axis, pupil, title in zip(
    axes,
    (circular_pupil, elt_pupil),
    ("Circular pupil", "ELT/HARMONI pupil"),
):
    image = axis.imshow(
        pupil.transmission.cpu(), origin="lower", extent=extent, cmap="gray"
    )
    axis.set_title(title)
    axis.set_xlabel("x [m]")
    axis.set_ylabel("y [m]")
    axis.set_aspect("equal")
fig.colorbar(image, ax=axes, shrink=0.8, label="Amplitude transmission")
plt.show()

## Weighted Gram matrices

For a command matrix \(B\), Fiatlux uses the normalized intensity metric

\[W = |P|^2 / \sum |P|^2, \qquad G = B^\mathsf{T} W B.\]

The sampled Zernikes are approximately orthonormal on their native circular disk. Central obscuration, segmentation and spiders destroy that orthogonality on the ELT pupil.

In [ ]:
def pupil_weights(pupil):
    weights = pupil.transmission.abs().square().flatten().to(grid.dtype)
    return weights / weights.sum()


def gram_matrix(command_matrix, pupil):
    weights = pupil_weights(pupil)
    return command_matrix.mT @ (weights[:, None] * command_matrix)


def max_off_diagonal(matrix):
    off_diagonal = matrix - torch.diag_embed(torch.diagonal(matrix))
    return float(off_diagonal.abs().max())

n_modes = 15
raw_basis = ZernikeBasis(grid, n=n_modes)
raw_matrix = raw_basis.build_command_matrix()
elt_basis = PupilOrthogonalizedBasis(raw_basis, elt_pupil)
elt_orthogonal_matrix = elt_basis.build_command_matrix()

gram_circular = gram_matrix(raw_matrix, circular_pupil)
gram_elt_raw = gram_matrix(raw_matrix, elt_pupil)
gram_elt_orthogonal = gram_matrix(elt_orthogonal_matrix, elt_pupil)
identity = torch.eye(n_modes, dtype=grid.dtype)

print(
    "Circular Zernike max |off-diagonal|:",
    f"{max_off_diagonal(gram_circular):.3e}",
)
print(
    "Raw ELT Zernike max |off-diagonal|:",
    f"{max_off_diagonal(gram_elt_raw):.3e}",
)
print(
    "Reorthogonalized ELT max |G-I|:",
    f"{float((gram_elt_orthogonal - identity).abs().max()):.3e}",
)

torch.testing.assert_close(
    gram_elt_orthogonal, identity, rtol=2e-10, atol=2e-10
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
matrices = (gram_circular, gram_elt_raw, gram_elt_orthogonal)
titles = (
    "Zernike / circular pupil",
    "Zernike / ELT pupil",
    "Reorthogonalized / ELT pupil",
)
for axis, matrix, title in zip(axes, matrices, titles):
    image = axis.imshow(
        matrix.cpu(), origin="lower", cmap="RdBu_r", vmin=-0.5, vmax=1.5
    )
    axis.set_title(title)
    axis.set_xlabel("Mode index")
    axis.set_ylabel("Mode index")
fig.colorbar(image, ax=axes, shrink=0.8, label=r"$B^T W B$")
plt.show()

## What happens to the ELT modes?

Löwdin orthogonalization mixes the original Zernikes while preserving their complete controlled subspace. The maps below are displayed only on transmitted ELT pixels. Each column uses the same color scale before and after orthogonalization.

In [ ]:
selected = [0, 1, 2, 3, 4, 10]
raw_modes = raw_matrix.mT.reshape(n_modes, grid.ny, grid.nx)
orthogonal_modes = elt_orthogonal_matrix.mT.reshape(n_modes, grid.ny, grid.nx)
elt_support = elt_pupil.transmission.to(grid.dtype)

fig, axes = plt.subplots(
    2, len(selected), figsize=(16, 6), constrained_layout=True
)
for column, mode_index in enumerate(selected):
    raw = raw_modes[mode_index] * elt_support
    orthogonal = orthogonal_modes[mode_index] * elt_support
    limit = float(torch.maximum(raw.abs().max(), orthogonal.abs().max()))
    for row, (mode, label) in enumerate(
        ((raw, "Raw Zernike"), (orthogonal, "ELT orthogonal"))
    ):
        image = axes[row, column].imshow(
            mode.cpu(),
            origin="lower",
            extent=extent,
            cmap="RdBu_r",
            vmin=-limit,
            vmax=limit,
        )
        axes[row, column].set_title(f"{label} Z{mode_index + 2}")
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
        axes[row, column].set_aspect("equal")
        fig.colorbar(image, ax=axes[row, column], shrink=0.72)
plt.show()

## Quantitative validation

Every reorthogonalized ELT mode has unit weighted RMS, all cross-products vanish to numerical precision, and the dtype/device remain those of the pupil grid.

In [ ]:
elt_rms = torch.sqrt(torch.diagonal(gram_elt_orthogonal))
print("ELT orthogonal mode RMS:", elt_rms.tolist())
print("dtype:", elt_orthogonal_matrix.dtype)
print("device:", elt_orthogonal_matrix.device)

torch.testing.assert_close(elt_rms, torch.ones_like(elt_rms), rtol=2e-10, atol=2e-10)
assert max_off_diagonal(gram_elt_orthogonal) < 2e-10
assert elt_orthogonal_matrix.dtype == grid.dtype
assert elt_orthogonal_matrix.device == grid.device